# ДЗ №3. Ранжирование на основе datamart

Общая информация
Дата выдачи: 3 апреля 2026

Дедлайн: 26 апреля 2026 23:59 MSK

В этом домашнем задании мы продолжим строить приближенную к реальной рекомендательную систему. Работать будем с данными marketplace из [T-ECD](https://huggingface.co/datasets/t-tech/T-ECD).

Обычно рекомендательная система состоит из нескольких этапов:
1. Отбор кандидатов (Retrieval)
2. Ранжирование (Ranking)
3. Бизнес-логика (например, условие на то, чтобы товары от одного продавца не стояли в ленте друг за другом)

В этом домашнем задании сосредоточимся на втором этапе. Можно и нужно использовать наработки из предыдущего домашнего задания!

Краткое напоминание, почему отбор кандидатов и ранжирование - разные этапы. Задачу рекомендаций можно решать как регрессию (насколько релевантен айтем), классификацию (релевантен ли айтем) или ранжирование (какой из двух айтемов релевантнее). В идеале - проранжировать каталог под каждого пользователя. Но каталог всегда существенно больше того подмножества айтемов, которые пользователь увидит в итоговой выдаче. А качественно ранжировать весь каталог - ОЧЕНЬ долго и дорого. Получаем trade-off скорости и качества. Простое решение - многостадийные рекомендации. Сначала отберем кандидатов (релевантные/не релевантные), а потом проранжируем только релевантные.

В этом задании следующая разбалловка:

1) Cбор датамарта - 4 балла
2) Сбор датасета для обучения ранжирования - 2 балла
3) Сбор град. бустинга и оценка по метрикам с бейзлайном - 4 балла

Соответственно, максимум можно набрать 10 баллов.

In [1]:
!pip install -q polars lightgbm scikit-learn catboost shap optuna implicit torch

In [3]:
import json
import gc
import os
import random
import typing as t
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import joblib
import lightgbm as lgbm
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import polars as pl
import polars.selectors as cs
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import HTML
from implicit.als import AlternatingLeastSquares
from optuna.samplers import TPESampler
from scipy.sparse import coo_matrix, csr_matrix
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset, IterableDataset



optuna.logging.set_verbosity(optuna.logging.WARNING)

## Download Data

Данные занимают около 3.5 GB

In [4]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/marketplace/**"]
)

/Users/vovazinev/Desktop/HSE/REC_HSE/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

'/Users/vovazinev/Desktop/HSE/REC_HSE/HW'

Больше всего места занимают эмбеддинги айтемов, их по желанию можно удалить, так как в этой работе нам потребуется дополнительное место на создание датамарта

In [5]:
pl.read_parquet("dataset/small/marketplace/items.pq").drop("embedding").write_parquet("dataset/small/marketplace/items.pq")

## I. Datamart (4 балла)

В задаче ранжирования хорошо себя показывают бустинги ([Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/)). С точки зрения интерфейса: Algorithm(user, item, [features]), где фичи - любые полезные статистики (количество просмотров, конверсия из клика в кликаут, средний рейтинг, ...).

Каждый раз рассчитывать фичи с нуля по сырым логам достаточно тяжело (а в реальности рекомендации мы делаем не один раз в домашней работе, а гораздо чаще). Учитывая, что логи могут иметь разный формат или нуждаться в  дополнительной фильтрации (баги в логах всё же не редкость). Простое решение - предподсчитать статистики и сохранить их в отдельных файлах, которые затем удобно просто прочитать. 

Это удобно сделать посредством датамарта. Датамарт - это витрина данных под определенную задачу. Он состоит из слоёв, где переход между слоями задает преобразование над данными, и обычно такие преобразования выполняются раз в какое-то время (в нашем случае пусть будет день). Для задачи ранжирования нам потребуется три слоя: 
1. Raw - содержит сырые логи за каждый день. Мы уже собрали его на предыдущем шаге.
2. Aggs - содержит агрегированные статистики по юзерам и айтемам за каждый день (user, item, [stats]). Например, количество просмотров, количество кликов, количество кликов c поверхности поиска.
3. Features - содержит фичи, которые мы хотим использовать в модели, тоже за каждый день. Например, средний рейтинг айтема, средний рейтинг айтема по категориям, общая конверсия из клика в кликаут для пользователя за последние 30 дней. На этом слое удобно выделить отдельные папки по группам фичей (user, item, user-item, ...), чтобы избежать дубликатов при хранении. 

Возьмем только небольшой срез данных, иначе дальнейшая работа может стать computationally infeasible. Переложим этот срез в `datamart/raw/events/{action_type}/{day}.pq`

Именно в таком формате логи обычно хранятся в сервисе. 

Вы можете расширить условия на сэмплирование юзеров и айтемов. Если у вас будут проблемы с памятью, то можете и уменьшить что-то, но чем меньше ваш датасет, тем хуже будут метрики у конечной модели

In [6]:
ACTION_TYPES = ["view", "click", "clickout", "like"]
SUBDOMAINS = ["u2i", "i2i", "catalog", "search", "other"]

DAYS = list(range(1250, 1301))  # не все даты

Будем считать, что view < click < clickout < like с точки зрения бизнеса. Этот факт будет использоваться далее.

In [7]:
selected_users = (
    pl.concat(
        [pl.scan_parquet(f"dataset/small/marketplace/events/{str(day).zfill(5)}.pq") for day in DAYS[-10:]]
    )
    .group_by("user_id").agg(pl.len()).sort("len", descending=True)
    .head(20000).collect()["user_id"].to_list()
)

selected_items = (
    pl.concat(
        [pl.scan_parquet(f"dataset/small/marketplace/events/{str(day).zfill(5)}.pq") for day in DAYS[-10:]]
    ).group_by("item_id").agg(pl.len()).sort("len", descending=True)
    .head(20000).collect()["item_id"].to_list()
)

In [8]:
USERS = pl.scan_parquet("dataset/small/users.pq").filter(pl.col("user_id").is_in(selected_users)).collect()
print(USERS.shape)
ITEMS = pl.scan_parquet("dataset/small/marketplace/items.pq").filter(pl.col("item_id").is_in(selected_items)).collect()

(20000, 3)


### I.I. Datamart -> Raw слой (1 из 4 баллов)

Здесь вам надо собрать  raw слой:

    datamart/
    ├── aggs/
    ├── features/
    └── raw/
        └── events/
            └── click/
                ├── 01200.pq
                ├── 01201.pq
                └── 01202.pq

в каждом файлике должны храниться данные на конкретную дату


In [9]:
raw_dir = Path("datamart/raw/events/")

for action_type in ACTION_TYPES:
    os.makedirs(raw_dir / action_type, exist_ok=True)

for day in tqdm(DAYS, desc="Building raw layer"):
    day_str = str(day).zfill(5)
    src_file = Path(f"dataset/small/marketplace/events/{day_str}.pq")
    if not src_file.exists():
        continue

    day_df = (
        pl.scan_parquet(src_file)
        .filter(
            pl.col("user_id").is_in(selected_users) &
            pl.col("item_id").is_in(selected_items)
        )
        .collect()
    )

    for action_type in ACTION_TYPES:
        out_path = raw_dir / action_type / f"{day_str}.pq"
        day_df.filter(pl.col("action_type") == action_type).write_parquet(out_path)

print("Raw layer done.")
print("Example files:")
for at in ACTION_TYPES:
    files = list((raw_dir / at).glob("*.pq"))
    print(f"  {at}: {len(files)} files")

Building raw layer:   0%|          | 0/51 [00:00<?, ?it/s]

Raw layer done.
Example files:
  view: 51 files
  click: 51 files
  clickout: 51 files
  like: 51 files


### I.II. Datamart -> Agg слой (1 из 4 баллов)

Рассчитате количество событий каждого типа (`action_type`) по каждой поверхности (`subdomain`) по парам (`user_id`, `item_id`) за каждый день. Не забудьте про общий счетчик - сумму по всем поверхностям. Сохраните в виде polars-таблиц.

Аналогично raw, но в agg значения внутри дня должны быть агрегированы

    datamart/
    └── aggs/
        └── events/
            ├── 01200.pq
            ├── 01201.pq
            └── 01202.pq

In [10]:
events_dir = Path("datamart/aggs/events/")
os.makedirs(events_dir, exist_ok=True)

for day in tqdm(DAYS, desc="Building agg layer"):
    day_str = str(day).zfill(5)

    frames = []
    for action_type in ACTION_TYPES:
        src = Path(f"datamart/raw/events/{action_type}/{day_str}.pq")
        if src.exists():
            frames.append(pl.read_parquet(src).with_columns(pl.lit(action_type).alias("action_type")))

    if not frames:
        continue

    day_df = pl.concat(frames)

    agg = (
        day_df
        .group_by(["user_id", "item_id", "action_type", "subdomain"])
        .agg(pl.len().alias("cnt"))
    )

    agg_wide = agg.pivot(
        on="subdomain",
        index=["user_id", "item_id", "action_type"],
        values="cnt",
        aggregate_function="sum",
    ).fill_null(0)

    # Rename columns to action_type__subdomain
    rename_map = {}
    for col in agg_wide.columns:
        if col not in ("user_id", "item_id", "action_type"):
            rename_map[col] = f"{col}"  

    # Add total column (sum across all subdomains)
    subdomain_cols = [c for c in agg_wide.columns if c not in ("user_id", "item_id", "action_type")]
    agg_wide = agg_wide.with_columns(
        pl.sum_horizontal(subdomain_cols).alias("total")
    )

    # Rename subdomain cols to include action_type prefix
    result_frames = []
    for at in ACTION_TYPES:
        at_df = agg_wide.filter(pl.col("action_type") == at).drop("action_type")
        rename = {c: f"{at}__{c}" for c in at_df.columns if c not in ("user_id", "item_id")}
        result_frames.append(at_df.rename(rename))

    result = result_frames[0]
    for rf in result_frames[1:]:
        result = result.join(rf, on=["user_id", "item_id"], how="full", coalesce=True)

    result = result.fill_null(0)
    result.write_parquet(events_dir / f"{day_str}.pq")

Building agg layer:   0%|          | 0/51 [00:00<?, ?it/s]

Пример того, что может получиться.

In [11]:
pl.read_parquet("datamart/aggs/events/01300.pq").sample(10)

user_id,item_id,view__catalog,view__u2i,view__i2i,view__other,view__search,view__null,view__total,click__catalog,click__u2i,click__i2i,click__other,click__search,click__null,click__total,clickout__catalog,clickout__u2i,clickout__i2i,clickout__other,clickout__search,clickout__null,clickout__total,like__catalog,like__u2i,like__i2i,like__other,like__search,like__null,like__total
u64,str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
9342066,"""nfmcg_27605604""",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10401180,"""nfmcg_17720161""",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
14277220,"""nfmcg_10491551""",0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
43349439,"""nfmcg_7105073""",0,1,0,1,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
68831502,"""nfmcg_27545147""",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
79163599,"""nfmcg_21899566""",1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
30671612,"""nfmcg_23660551""",0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
72353832,"""nfmcg_23389912""",0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
77311937,"""nfmcg_11353763""",0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### I.III. Datamart -> Feature слой (2 из 4 баллов)

Подумайте, какие признаки, рассчитанные на основе ранее собранных статистик, можно будет использовать в качестве фичей для модели. Реализуйте логику их подсчета за каждый день. Обратите внимание, что все фичи необходимо рассчитывать по какому-то временному окну, например "количество кликов на товар за последние 14 дней".

Сохраните признаки в виде polars-таблиц. Не забудьте про декомпозицию на отдельные папки по сущностям с целью избежать дубликатов при хранении.

Минимально должно получиться 10 признаков, из которых:
* 2 должны относиться только к сущности "пользователь" (например, медианная цена кликнутых айтемов у этого пользователя)
* 2 должны относиться только к сущности "айтем" (например, средняя конверсия из клика в кликаут по поверхности "поиск" у этого айтема)
* 6 признаков, которые показывают связь пользователя и айтема (например, количество просмотров этого айтема у этого пользователя).

Однако настоятельно рекомендуется собрать больше признаков. В данных много сущностей - категория, соцдем-кластер, бренд. И много параметров - цена, поверхность, тип события. В том числе можно рассчитать "изменение признака относительно предыдущего дня". Используйте polars expressions для написания шаблонного кода, в который затем удобно подставить конкретные названия сущностей и параметров, и получить готовый набор признаков. 

P.S. Цену для товаров мы считаем фиксированной, она представлена в каталоге `ITEMS`

Может быть полезно посчитать конверсии как отношение числа второго события из пары к числу первого события из пары. Аккуратнее с делением на ноль - возможно, пользователей, для которых не определено число первого события из пары, не стоит учитывать при расчетах.

Пример того, как должно получиться:



        datamart/
        ├── aggs/
        └── features/
            └── events/
                ├── brand_id/
                ├── category//
                ├── item_id/
                ├── socdem_cluster-brand_id/
                ├── socdem_cluster-category/
                └── socdem_cluster-item_id/
                    ├── 01200.pq
                    ├── 01201.pq
                    ├── 01202.pq
                    └── 01203.pq

In [12]:
CONVERSION_PAIRS = [
    ("view", "click"), 
    ("view", "clickout"), 
    ("view", "like"),
    ("click", "clickout"),
    ("click", "like"),
    ("clickout", "like"),
]

Удобно называть фичи следующим образом.

In [13]:
def _feature_name(
    feature: str,
    keys: list[str],
    type: t.Literal["num", "cat"] = "num",
) -> str:
    return f"f_num__{'_'.join(keys)}__{feature}"

In [14]:
def _load_window_agg(day: int, num_days: int, day_from: int) -> pl.LazyFrame | None:
    """Load and concat agg files for window [day-num_days, day-1]."""
    agg_dir = Path("datamart/aggs/events/")
    frames = []
    for d in range(max(day_from, day - num_days), day):
        f = agg_dir / f"{str(d).zfill(5)}.pq"
        if f.exists():
            frames.append(pl.scan_parquet(f))
    if not frames:
        return None
    return pl.concat(frames)


def _build_count_exprs(available_cols: list[str], group_keys: list[str], num_days: int) -> list[pl.Expr]:
    """Build sum expressions for all available action_type x subdomain columns."""
    exprs = []
    for at in ACTION_TYPES:
        total_col = f"{at}__total"
        if total_col in available_cols:
            exprs.append(
                pl.col(total_col).sum().alias(
                    _feature_name(f"cnt_{at}_{num_days}d", group_keys)
                )
            )
        for sd in SUBDOMAINS:
            col = f"{at}__{sd}"
            if col in available_cols:
                exprs.append(
                    pl.col(col).sum().alias(
                        _feature_name(f"cnt_{at}_{sd}_{num_days}d", group_keys)
                    )
                )
    return exprs


def _add_conversion_features(result: pl.DataFrame, group_keys: list[str], num_days: int) -> pl.DataFrame:
    """Add conversion ratio features (b/a) for each CONVERSION_PAIR."""
    for (at_a, at_b) in CONVERSION_PAIRS:
        col_a = _feature_name(f"cnt_{at_a}_{num_days}d", group_keys)
        col_b = _feature_name(f"cnt_{at_b}_{num_days}d", group_keys)
        conv_name = _feature_name(f"conv_{at_a}_to_{at_b}_{num_days}d", group_keys)
        if col_a in result.columns and col_b in result.columns:
            result = result.with_columns(
                (pl.col(col_b) / pl.col(col_a).replace(0, None)).alias(conv_name)
            )
    return result


def calculate_user_group_item_group_features(
    day_from: int = 1250,
    day_to: int = 1300,
    num_days: int = 14,
    user_group: t.Literal["region", "socdem_cluster"] | None = None,
    item_group: t.Literal["brand_id", "category", "subcategory", "item_id"] = "item_id"
) -> None:
    """
    Calculate features aggregated by (user_group, item_group) over a rolling window.
    Saves one parquet per day to datamart/features/events/{dir_name}/

    If user_group is None -> aggregates only by item_group (item-only features).
    day_from should match min(DAYS) so agg files actually exist.
    """
    global USERS, ITEMS

    dir_name = f"{user_group}-{item_group}" if user_group is not None else item_group
    out_dir = Path(f"datamart/features/events/{dir_name}/")
    out_dir.mkdir(parents=True, exist_ok=True)

    item_cols = ["item_id"] + ([item_group] if item_group != "item_id" else [])
    items_lf = ITEMS.select(item_cols).lazy()

    users_lf = None
    if user_group is not None:
        users_lf = USERS.select(["user_id", user_group]).lazy()

    for day in range(day_from, day_to + 1):
        day_str = str(day).zfill(5)
        out_path = out_dir / f"{day_str}.pq"

        agg_lf = _load_window_agg(day, num_days, day_from)
        if agg_lf is None:
            continue

        # Join item_group
        agg_lf = agg_lf.join(items_lf, on="item_id", how="left")

        # Join user_group
        if users_lf is not None:
            agg_lf = agg_lf.join(users_lf, on="user_id", how="left")

        # Group keys
        group_keys = []
        if user_group is not None:
            group_keys.append(user_group)
        group_keys.append(item_group)

        available_cols = agg_lf.collect_schema().names()
        count_exprs = _build_count_exprs(available_cols, group_keys, num_days)
        if not count_exprs:
            continue

        result = agg_lf.group_by(group_keys).agg(count_exprs).collect()
        result = _add_conversion_features(result, group_keys, num_days)
        result.write_parquet(out_path)


def calculate_user_item_group_features(
    day_from: int = 1250,
    day_to: int = 1300,
    num_days: int = 14,
    item_group: t.Literal["item_id", "brand_id", "category", "subcategory"] | None = None
) -> None:
    """
    Calculate features aggregated by (user_id [, item_group]) over a rolling window.
    If item_group is None -> user-only features (aggregated by user_id only).
    Saves one parquet per day to datamart/features/events/{dir_name}/
    day_from should match min(DAYS) so agg files actually exist.
    """
    global USERS, ITEMS

    dir_name = f"user_id-{item_group}" if item_group is not None else "user_id"
    out_dir = Path(f"datamart/features/events/{dir_name}/")
    out_dir.mkdir(parents=True, exist_ok=True)

    items_lf = None
    if item_group is not None and item_group != "item_id":
        items_lf = ITEMS.select(["item_id", item_group]).lazy()


    price_lf = ITEMS.select(["item_id", "price"]).lazy()

    for day in range(day_from, day_to + 1):
        day_str = str(day).zfill(5)
        out_path = out_dir / f"{day_str}.pq"

        agg_lf = _load_window_agg(day, num_days, day_from)
        if agg_lf is None:
            continue

        if items_lf is not None:
            agg_lf = agg_lf.join(items_lf, on="item_id", how="left")

        group_keys = ["user_id"]
        if item_group is not None:
            group_keys.append(item_group)

        available_cols = agg_lf.collect_schema().names()
        count_exprs = _build_count_exprs(available_cols, group_keys, num_days)


        if item_group is None:
            click_col = "click__total"
            if click_col in available_cols:
                price_stats = (
                    agg_lf
                    .group_by(["user_id", "item_id"])
                    .agg(pl.col(click_col).sum().alias("total_clicks_window"))
                    .filter(pl.col("total_clicks_window") > 0)
                    .join(price_lf, on="item_id", how="left")
                    .group_by("user_id")
                    .agg([
                        pl.col("price").median().alias(
                            _feature_name(f"median_price_clicked_{num_days}d", ["user_id"])
                        ),
                        pl.col("price").mean().alias(
                            _feature_name(f"mean_price_clicked_{num_days}d", ["user_id"])
                        ),
                    ])
                    .collect()
                )

                if not count_exprs:
                    price_stats.write_parquet(out_path)
                    continue

                result = agg_lf.group_by(group_keys).agg(count_exprs).collect()
                result = _add_conversion_features(result, group_keys, num_days)
                result = result.join(price_stats, on="user_id", how="left")
                result.write_parquet(out_path)
                continue

        if not count_exprs:
            continue

        result = agg_lf.group_by(group_keys).agg(count_exprs).collect()
        result = _add_conversion_features(result, group_keys, num_days)
        result.write_parquet(out_path)

In [15]:
for item_group in [None, "item_id", "brand_id", "category"]:
    print(f"Calculating features for user_id and {item_group}")
    calculate_user_item_group_features(
        day_from=min(DAYS),
        day_to=max(DAYS),
        num_days=30,
        item_group=item_group,
    )

Calculating features for user_id and None
Calculating features for user_id and item_id
Calculating features for user_id and brand_id
Calculating features for user_id and category


In [16]:
for user_group in [None, "socdem_cluster"]:
    for item_group in ["item_id", "brand_id", "category"]:
        print(f"Calculating features for {user_group} and {item_group}")
        calculate_user_group_item_group_features(
            day_from=min(DAYS),
            day_to=max(DAYS),
            num_days=30,
            user_group=user_group,
            item_group=item_group,
        )

Calculating features for None and item_id
Calculating features for None and brand_id
Calculating features for None and category
Calculating features for socdem_cluster and item_id
Calculating features for socdem_cluster and brand_id
Calculating features for socdem_cluster and category


Пример того, что может получиться.

In [17]:
!ls datamart/features/events

brand_id                socdem_cluster-category user_id-category
category                socdem_cluster-item_id  user_id-item_id
item_id                 user_id
socdem_cluster-brand_id user_id-brand_id


In [18]:
#01250 нету чтобы избежать data leakage
pl.read_parquet("datamart/features/events/user_id/01251.pq").sample(10)

user_id,f_num__user_id__cnt_view_30d,f_num__user_id__cnt_view_u2i_30d,f_num__user_id__cnt_view_i2i_30d,f_num__user_id__cnt_view_catalog_30d,f_num__user_id__cnt_view_search_30d,f_num__user_id__cnt_view_other_30d,f_num__user_id__cnt_click_30d,f_num__user_id__cnt_click_u2i_30d,f_num__user_id__cnt_click_i2i_30d,f_num__user_id__cnt_click_catalog_30d,f_num__user_id__cnt_click_search_30d,f_num__user_id__cnt_click_other_30d,f_num__user_id__cnt_clickout_30d,f_num__user_id__cnt_clickout_u2i_30d,f_num__user_id__cnt_clickout_i2i_30d,f_num__user_id__cnt_clickout_catalog_30d,f_num__user_id__cnt_clickout_search_30d,f_num__user_id__cnt_clickout_other_30d,f_num__user_id__cnt_like_30d,f_num__user_id__cnt_like_u2i_30d,f_num__user_id__cnt_like_i2i_30d,f_num__user_id__cnt_like_catalog_30d,f_num__user_id__cnt_like_search_30d,f_num__user_id__cnt_like_other_30d,f_num__user_id__conv_view_to_click_30d,f_num__user_id__conv_view_to_clickout_30d,f_num__user_id__conv_view_to_like_30d,f_num__user_id__conv_click_to_clickout_30d,f_num__user_id__conv_click_to_like_30d,f_num__user_id__conv_clickout_to_like_30d,f_num__user_id__median_price_clicked_30d,f_num__user_id__mean_price_clicked_30d
u64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64
34177366,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null
87036714,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null
8194552,2,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null
66839506,19,15,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null
9956850,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null
14368941,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null
87382087,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null
17723409,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null
56093939,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,null,null,null,null,null


## II. Сбор датасета. (3 балла)

Будем учить модель ранжировать показанные пользователю рекомендации в рамках дня (можно было бы выбрать и другой промежуток). Разделим подготовку обучающих данных на два этапа: сбор "скелета" (базиса) с последующим созданием датасета путем джойна фичей на базис.

### II.I Сбор датасета -> Сбор базиса (1 из 3 баллов)

Базис представляется как (`session_id`, `user_id`, `item_id`, `label`), где в качестве `session_id` используется конкатенация `user_id` и `day`, а `label` зависит от `action_type`. Напомню, что view < click < clickout < like. Сессии, целиком состоящие из view, не стоит учитывать (действительно, сложно оценить качество сортировки одинаковых элементов). Если в рамках сессии было несколько взаимодействий с айтемом, то в качестве `label` нужно взять максимальное значение.

После того, как получим предсказания модели, можно будет сгруппировать базис по `session_id` и получить структуру ([`item_id`], [`label`], [`score`]) - тогда, отсортировав по айтемы по `label` либо `score` получим список айтемов с реальной либо модельной сортировкой, а по этому уже удобно считать метрики.

Реализуйте сбор базиса. Его так же удобно сохранять "за каждый день". Добавьте логику фильтрации сессий по 99 перцентилю длины. 

In [19]:
def build_basis(
    day_from: int,
    day_to: int,
    filter_99: bool = True,
    output_dir: Path = Path("output/basis/"),
) -> None:
    """
    Sessions consisting entirely of views (max label == 0) are dropped.
    If filter_99=True, sessions longer than 99th percentile are dropped.
    """
    raw_dir = Path("datamart/raw/events/")
    output_dir.mkdir(parents=True, exist_ok=True)

    label_map = {"view": 0, "click": 1, "clickout": 2, "like": 3}

    for day in tqdm(range(day_from, day_to + 1), desc="Building basis"):
        day_str = str(day).zfill(5)
        out_path = output_dir / f"{day_str}.pq"

        frames = []
        for at, lbl in label_map.items():
            src = raw_dir / at / f"{day_str}.pq"
            if src.exists():
                frames.append(
                    pl.read_parquet(src)
                    .select(["user_id", "item_id"])
                    .with_columns(pl.lit(lbl).alias("label"))
                )

        if not frames:
            continue

        day_df = pl.concat(frames)

        # session_id = user_id + "_" + day
        day_df = day_df.with_columns(
            (pl.col("user_id").cast(pl.Utf8) + "_" + pl.lit(str(day))).alias("session_id")
        )

        # For each (session_id, item_id) keep max label
        day_df = (
            day_df
            .group_by(["session_id", "user_id", "item_id"])
            .agg(pl.col("label").max())
        )

        # Drop sessions where ALL items have label == 0 (view-only sessions)
        session_max = (
            day_df
            .group_by("session_id")
            .agg(pl.col("label").max().alias("session_max_label"))
        )
        day_df = day_df.join(session_max, on="session_id", how="left")
        day_df = day_df.filter(pl.col("session_max_label") > 0).drop("session_max_label")

        if len(day_df) == 0:
            continue

        # Filter by 99th percentile of session length
        if filter_99:
            session_len = (
                day_df
                .group_by("session_id")
                .agg(pl.len().alias("session_len"))
            )
            p99 = session_len["session_len"].quantile(0.99)
            valid_sessions = session_len.filter(pl.col("session_len") <= p99)["session_id"]
            day_df = day_df.filter(pl.col("session_id").is_in(valid_sessions))

        day_df.write_parquet(out_path)

In [20]:
build_basis(day_from=min(DAYS), day_to=max(DAYS), filter_99=True)

Building basis:   0%|          | 0/51 [00:00<?, ?it/s]

/var/folders/29/7x6m0y55313f2bfw3451j0780000gn/T/ipykernel_23107/3679383202.py:68: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  day_df = day_df.filter(pl.col("session_id").is_in(valid_sessions))


In [21]:
basis = pl.read_parquet("output/basis/")
print(basis.shape)
basis.sample(10)

(1139772, 4)


session_id,user_id,item_id,label
str,u64,str,i32
"""73864292_1268""",73864292,"""nfmcg_16621063""",0
"""74232349_1292""",74232349,"""nfmcg_21074179""",0
"""58004598_1290""",58004598,"""nfmcg_20867318""",0
"""14545569_1296""",14545569,"""nfmcg_25338217""",0
"""72253304_1288""",72253304,"""nfmcg_20867318""",1
"""39255154_1299""",39255154,"""nfmcg_2783866""",0
"""60229593_1288""",60229593,"""nfmcg_28197851""",0
"""40863761_1298""",40863761,"""nfmcg_20346711""",0
"""70836860_1284""",70836860,"""nfmcg_10232568""",0


In [22]:
for col in basis.columns:
    print(f"{col}: {basis[col].n_unique()}")

session_id: 38223
user_id: 15875
item_id: 19968
label: 4


In [23]:
basis["label"].value_counts().sort("label")

label,count
i32,u32
0,1036525
1,86353
2,15077
3,1817


In [24]:
basis.group_by("session_id").agg(pl.len())["len"].describe()

statistic,value
str,f64
"""count""",38223.0
"""null_count""",0.0
"""mean""",29.819009
"""std""",31.027197
"""min""",1.0
"""25%""",8.0
"""50%""",20.0
"""75%""",40.0
"""max""",204.0


### II.II Сбор датасета -> Сбор датасета с фичами (2 из 3 баллов)

Чтобы создать датасет, достаточно приджойнить к базису фичи. Обратите внимание, что фичи должны быть собраны за предыдущий день, чтобы избежать ликов. То есть, если мы работаем с базисом на 1300 день, то фичи для него необходимо брать из 1299 дня. Помимо числовых, можно также добавить категориальные фичи.

Реализуйте необходимую логику. Добавьте возможность читать список фичей, которые необходимо приджойнить, из файла. 

> В условии  сказано: "фичи для 1300 дня необходимо брать из 1299 дня, чтобы избежать ликов".

    Однако в моей реализации явный сдвиг feat_day = day - 1 на этапе сборки датасета не требуется. Защита от Data Leakage  уже  реализована "под капотом" на этапе построения Datamart.

    При генерации фичей (Part I) для файла 01300.pq агрегация событий происходила по циклу range(1300 - num_days, 1300). В Python этот диапазон берет данные строго по 1299-й день включительно. Таким образом, файл фичей 01300.pq уже содержит актуальную историю до вчерашнего дня и физически не включает события сегодняшнего (1300) дня.


In [25]:
def join_features(
    df: pl.LazyFrame,
    day: int,
    users: pl.LazyFrame,
    items: pl.LazyFrame,
    features_dir: Path = Path("datamart/features/events/"),
    features_to_use: list[str] | None = None,
) -> pl.LazyFrame:
    """
    Join all available feature tables for `day` onto the basis LazyFrame.
    No day-1 shift needed here: feature file `{day}.pq` is already built from
    window [day-num_days, day-1], so leakage is prevented by construction in Part I.
    """
    feat_day = day
    feat_day_str = str(feat_day).zfill(5)


    df = df.join(users, on="user_id", how="left")
    df = df.join(items, on="item_id", how="left")


    if not features_dir.exists():
        return df

    for feat_subdir in sorted(features_dir.iterdir()):
        if not feat_subdir.is_dir():
            continue

        feat_file = feat_subdir / f"{feat_day_str}.pq"
        if not feat_file.exists():
            continue

        feat_df = pl.scan_parquet(feat_file)
        feat_schema = feat_df.collect_schema()
        feat_cols = feat_schema.names()

        if features_to_use is not None:
            keep_cols = [c for c in feat_cols if c in features_to_use or not c.startswith("f_")]
            feat_df = feat_df.select(keep_cols)
            feat_cols = keep_cols

        # Determine join keys from directory name
        dir_name = feat_subdir.name  #  "user_id", "item_id", "user_id-item_id", "socdem_cluster-brand_id"
        parts = dir_name.split("-")

        # Map directory parts to actual column names
        join_keys = []
        valid = True
        for part in parts:
            if part in feat_cols:
                join_keys.append(part)
            else:
                valid = False
                break

        if not valid or not join_keys:
            continue

        # Only keep feature columns (not join keys) from feat_df to avoid duplicates
        feature_only_cols = [c for c in feat_cols if c not in join_keys]
        if not feature_only_cols:
            continue

        feat_df = feat_df.select(join_keys + feature_only_cols)


        df_schema = df.collect_schema().names()
        if not all(k in df_schema for k in join_keys):
            continue

        df = df.join(feat_df, on=join_keys, how="left")

    return df

In [26]:
def build_dataset(
    day_from: int,
    day_to: int,
    basis_dir: Path = Path("output/basis/"),
    output_dir: Path = Path("output/dataset/"),
    features_to_use_filepath: Path | None = None,
    users: pl.LazyFrame | None = None,
    items: pl.LazyFrame | None = None,
) -> None:
    """
    Build dataset by joining features onto basis for each day.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    features_dir = Path("datamart/features/events/")

    # Load features_to_use list if provided
    features_to_use = None
    if features_to_use_filepath is not None and features_to_use_filepath.exists():
        with open(features_to_use_filepath) as f:
            features_to_use = [line.strip() for line in f if line.strip()]

    # Default user/item metadata
    if users is None:
        users = USERS.select(["user_id", "socdem_cluster", "region"]).lazy()
    if items is None:
        items = ITEMS.select(["item_id", "category", "subcategory", "brand_id", "price"]).lazy()

    for day in tqdm(range(day_from, day_to + 1), desc="Building dataset"):
        day_str = str(day).zfill(5)
        basis_file = basis_dir / f"{day_str}.pq"
        out_path = output_dir / f"{day_str}.pq"

        if not basis_file.exists():
            continue

        basis_lf = pl.scan_parquet(basis_file)

        # Join features (from day-1)
        dataset_lf = join_features(
            df=basis_lf,
            day=day,
            users=users,
            items=items,
            features_dir=features_dir,
            features_to_use=features_to_use,
        )

        dataset = dataset_lf.collect()

        # Fill nulls in feature columns with 0
        feat_cols = [c for c in dataset.columns if c.startswith("f_")]
        if feat_cols:
            dataset = dataset.with_columns([
                pl.col(c).fill_null(0) for c in feat_cols
            ])

        dataset.write_parquet(out_path)

In [27]:
build_dataset(day_from=min(DAYS), day_to=max(DAYS), output_dir = Path("output/dataset/"))

Building dataset:   0%|          | 0/51 [00:00<?, ?it/s]

In [28]:
pl.read_parquet("output/dataset/01251.pq").sample(10)

session_id,user_id,item_id,label,socdem_cluster,region,category,subcategory,brand_id,price,f_num__brand_id__cnt_view_30d,f_num__brand_id__cnt_view_u2i_30d,f_num__brand_id__cnt_view_i2i_30d,f_num__brand_id__cnt_view_catalog_30d,f_num__brand_id__cnt_view_search_30d,f_num__brand_id__cnt_view_other_30d,f_num__brand_id__cnt_click_30d,f_num__brand_id__cnt_click_u2i_30d,f_num__brand_id__cnt_click_i2i_30d,f_num__brand_id__cnt_click_catalog_30d,f_num__brand_id__cnt_click_search_30d,f_num__brand_id__cnt_click_other_30d,f_num__brand_id__cnt_clickout_30d,f_num__brand_id__cnt_clickout_u2i_30d,f_num__brand_id__cnt_clickout_i2i_30d,f_num__brand_id__cnt_clickout_catalog_30d,f_num__brand_id__cnt_clickout_search_30d,f_num__brand_id__cnt_clickout_other_30d,f_num__brand_id__cnt_like_30d,f_num__brand_id__cnt_like_u2i_30d,f_num__brand_id__cnt_like_i2i_30d,f_num__brand_id__cnt_like_catalog_30d,f_num__brand_id__cnt_like_search_30d,f_num__brand_id__cnt_like_other_30d,f_num__brand_id__conv_view_to_click_30d,f_num__brand_id__conv_view_to_clickout_30d,f_num__brand_id__conv_view_to_like_30d,…,f_num__user_id_category__cnt_like_other_30d,f_num__user_id_category__conv_view_to_click_30d,f_num__user_id_category__conv_view_to_clickout_30d,f_num__user_id_category__conv_view_to_like_30d,f_num__user_id_category__conv_click_to_clickout_30d,f_num__user_id_category__conv_click_to_like_30d,f_num__user_id_category__conv_clickout_to_like_30d,f_num__user_id_item_id__cnt_view_30d,f_num__user_id_item_id__cnt_view_u2i_30d,f_num__user_id_item_id__cnt_view_i2i_30d,f_num__user_id_item_id__cnt_view_catalog_30d,f_num__user_id_item_id__cnt_view_search_30d,f_num__user_id_item_id__cnt_view_other_30d,f_num__user_id_item_id__cnt_click_30d,f_num__user_id_item_id__cnt_click_u2i_30d,f_num__user_id_item_id__cnt_click_i2i_30d,f_num__user_id_item_id__cnt_click_catalog_30d,f_num__user_id_item_id__cnt_click_search_30d,f_num__user_id_item_id__cnt_click_other_30d,f_num__user_id_item_id__cnt_clickout_30d,f_num__user_id_item_id__cnt_clickout_u2i_30d,f_num__user_id_item_id__cnt_clickout_i2i_30d,f_num__user_id_item_id__cnt_clickout_catalog_30d,f_num__user_id_item_id__cnt_clickout_search_30d,f_num__user_id_item_id__cnt_clickout_other_30d,f_num__user_id_item_id__cnt_like_30d,f_num__user_id_item_id__cnt_like_u2i_30d,f_num__user_id_item_id__cnt_like_i2i_30d,f_num__user_id_item_id__cnt_like_catalog_30d,f_num__user_id_item_id__cnt_like_search_30d,f_num__user_id_item_id__cnt_like_other_30d,f_num__user_id_item_id__conv_view_to_click_30d,f_num__user_id_item_id__conv_view_to_clickout_30d,f_num__user_id_item_id__conv_view_to_like_30d,f_num__user_id_item_id__conv_click_to_clickout_30d,f_num__user_id_item_id__conv_click_to_like_30d,f_num__user_id_item_id__conv_clickout_to_like_30d
str,u64,str,i32,u8,u8,str,str,u64,f64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,…,u32,f64,f64,f64,f64,f64,f64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64
"""51157619_1251""",51157619,"""nfmcg_14384356""",0,9,32,"""Electronic Devices and Gadgets""","""TV Panels and Components""",137356,2.795409,3219,1972,220,504,320,203,126,58,16,30,22,0,15,0,3,1,9,2,1,0,0,0,0,0,0.039143,0.00466,0.000311,…,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
"""34126680_1251""",34126680,"""nfmcg_24544811""",0,19,37,"""Cosmetics, Personal Care, and …","""Body Skin Hygiene and Care Pro…",211031,-0.24633,1297,598,29,382,192,96,34,8,2,4,20,0,25,1,1,1,18,4,0,0,0,0,0,0,0.026214,0.019275,0.0,…,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
"""3002528_1251""",3002528,"""nfmcg_5083489""",0,12,29,"""Electronic Devices and Gadgets""","""Portable Audio Electronics""",137356,2.226142,3219,1972,220,504,320,203,126,58,16,30,22,0,15,0,3,1,9,2,1,0,0,0,0,0,0.039143,0.00466,0.000311,…,0,0.055556,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0

## III Обучение ранжирования (4 балла)

### III.I. Подготовка выборок для обучения/валидации/теста и реализация метрик (1 из 4 баллов)

Полезно будет также написать функцию, считывающую с диска и возвращающую train, val, train_val, и test части датасета. Диапазон будем задавать через дни.

In [29]:
def read_dataset(
    day_from: int,
    n_train_days: int,
    n_val_days: int,
    n_test_days: int,
    dataset_dir: Path = Path("output/dataset/"),
) -> tuple[pl.LazyFrame, pl.LazyFrame, pl.LazyFrame, pl.LazyFrame]:

    train_end     = day_from + n_train_days
    val_end       = train_end + n_val_days
    test_end      = val_end + n_test_days

    def _load(day_start: int, day_stop: int) -> pl.LazyFrame:
        frames = []
        for day in range(day_start, day_stop):
            f = dataset_dir / f"{str(day).zfill(5)}.pq"
            if f.exists():
                frames.append(pl.scan_parquet(f))
        if not frames:
            return pl.LazyFrame()
        return pl.concat(frames)

    train_lf     = _load(day_from, train_end)
    val_lf       = _load(train_end, val_end)
    train_val_lf = _load(day_from, val_end)
    test_lf      = _load(val_end, test_end)

    return train_lf, val_lf, train_val_lf, test_lf

In [30]:
train_df, val_df, train_val_df, test_df = read_dataset(
    day_from=1280,  
    n_train_days=14,
    n_val_days=1,
    n_test_days=1,
    dataset_dir=Path("output/dataset/")
)
train_df = train_df.collect()
val_df = val_df.collect()
train_val_df = train_val_df.collect()
test_df = test_df.collect()

> беру `day_from` = 1280 а не 1265 так как есть признаки которые считаются по окну за 30 дней. Также мы брали лишь топ самых активных пользователей за последние 10 дней. Переход с 1265 на 1280 увеличил датасет, так как повысилась вероятность встретить тех самых активных пользователей

In [31]:
train_val_df.shape

(466661, 312)

In [32]:
test_df.shape

(69541, 312)

Реализуйте логику расчета метрик по структуре ([`item_id`], [`label`], [`score`]) - можете формировать эту структуру также внутри функции расчета метрик, а можете вне. В качестве метрики обязательно использовать NDCG@k. В выборе остальных метрик вы свободны. Полезно может быть считать метрики в разрезе по label - например, сколько айтемов с label=2 попали в топ-10 рекомендаций.

Посчитайте метрики в A/A-сеттинге.

In [33]:
class AtKMetric(ABC):
    def __init__(self, k: int):
        self.k = k

    @property
    @abstractmethod
    def name(self) -> str:
        raise NotImplementedError

    @property
    def full_name(self) -> str:
        return f"{self.name}@{self.k}"

    @abstractmethod
    def __call__(self, *, labels_col: str = "labels", targets_col: str = "targets") -> pl.Expr:
        raise NotImplementedError


class NdcgAtK(AtKMetric):
    @property
    def name(self) -> str:
        return "ndcg"

    def __call__(self, *, labels_col: str = "preds", targets_col: str = "targets") -> pl.Expr:
        """
        preds_col:   list of structs {item_id} sorted by predicted score desc
        targets_col: list of structs {item_id, label} sorted by label desc (ideal order)
        """
        k = self.k

        def _ndcg_row(row):
            preds_list   = row[labels_col]    # list of {item_id: ...}
            targets_list = row[targets_col]   # list of {item_id: ..., label: ...}

            pred_ids   = [p["item_id"] for p in preds_list[:k]]
            target_map = {t["item_id"]: t["label"] for t in targets_list}

            # DCG
            dcg = 0.0
            for i, item_id in enumerate(pred_ids):
                rel = target_map.get(item_id, 0)
                if rel > 0:
                    dcg += (2.0 ** rel - 1.0) / np.log2(i + 2.0)

            # IDCG: ideal ordering
            ideal_labels = sorted(target_map.values(), reverse=True)[:k]
            idcg = sum(
                (2.0 ** r - 1.0) / np.log2(i + 2.0)
                for i, r in enumerate(ideal_labels)
                if r > 0
            )

            if idcg == 0.0:
                return 0.0
            return dcg / idcg

        return (
            pl.struct([labels_col, targets_col])
            .map_elements(_ndcg_row, return_dtype=pl.Float64)
        )


def evaluate_ranker(
    df: pl.DataFrame,
    ks: list[int] = [1, 5, 10, 20, 50],
    preds_col: str = "preds",
    targets_col: str = "targets",
) -> dict:
    """
    preds   (list of structs {item_id}, sorted by predicted score desc),
    targets (list of structs {item_id, label}, sorted by label desc)

    Returns a dict {metric_name: float}.
    """
    result = {}
    for k in ks:
        metric = NdcgAtK(k)
        expr = metric(labels_col=preds_col, targets_col=targets_col)
        col_name = metric.full_name
        result[col_name] = df.select(expr.alias(col_name))[col_name].mean()
    return result

In [34]:
metrics_best = evaluate_ranker(
    test_df.with_columns(score=pl.col("label"))
    .group_by("session_id").agg(
        [
            pl.struct("item_id").sort_by("score", descending=True).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)
metrics_worst = evaluate_ranker(
    test_df.with_columns(score=pl.col("label"))
    .group_by("session_id").agg(
        [
            pl.struct("item_id").sort_by("score", descending=False).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)
RESULTS = pd.concat([
    pd.DataFrame(metrics_best, index=["best"]),
    pd.DataFrame(metrics_worst, index=["worst"]), 
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.06050,0.09788,0.13197,0.18067,0.27392


### III.II. Реализация TopPopular бейзлайна (1 из 4 баллов)

Реализуйте любой бейзлайн (бейзлайны) на своё усмотрение. Посчитайте метрики. Не забудьте про консистентность: учимся на train - оцениваем на val; учимся на train+val - оцениваем на test.

Важно: используйте `sample(fraction=1.0, shuffle=True)` при группировке по сессии для расчета метрик, чтобы в случае одинаковых скоров автоматом не проставлялся скор из корркетно отсортированной последовтаельности айтемов! 

In [35]:
# TopPopular baseline: score = total click count of item in train

item_pop_train = (
    train_df
    .group_by("item_id")
    .agg(pl.col("label").sum().alias("pop_score"))
)


val_with_score = (
    val_df
    .join(item_pop_train, on="item_id", how="left")
    .with_columns(pl.col("pop_score").fill_null(0))
)

# Use random_noise as tie-breaker to avoid hash-join order artifacts
val_grouped = (
    val_with_score
    .with_columns(pl.lit(np.random.default_rng(42).random(val_with_score.height)).alias("random_noise"))
    .group_by("session_id")
    .agg([
        pl.struct("item_id").sort_by(["pop_score", "random_noise"], descending=[True, True]).alias("preds"),
        pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
    ])
)

baseline_val = evaluate_ranker(val_grouped)
print("Baseline (TopPop) on val:", baseline_val)

item_pop_train_val = (
    train_val_df
    .group_by("item_id")
    .agg(pl.col("label").sum().alias("pop_score"))
)


test_with_score = (
    test_df
    .join(item_pop_train_val, on="item_id", how="left")
    .with_columns(pl.col("pop_score").fill_null(0))
)

test_grouped = (
    test_with_score
    .with_columns(pl.lit(np.random.default_rng(42).random(test_with_score.height)).alias("random_noise"))
    .group_by("session_id")
    .agg([
        pl.struct("item_id").sort_by(["pop_score", "random_noise"], descending=[True, True]).alias("preds"),
        pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
    ])
)

baseline = evaluate_ranker(test_grouped)
print("Baseline (TopPop) on test:", baseline)

RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(baseline, index=["baseline"]),
])

RESULTS.style.format(precision=5).background_gradient(cmap="Blues")


Baseline (TopPop) on val: {'ndcg@1': 0.25048068519489597, 'ndcg@5': 0.3552419595003899, 'ndcg@10': 0.4229855200683765, 'ndcg@20': 0.4824302420326067, 'ndcg@50': 0.5282675056368475}
Baseline (TopPop) on test: {'ndcg@1': 0.24465786314525811, 'ndcg@5': 0.36097426265174765, 'ndcg@10': 0.4282719910116878, 'ndcg@20': 0.48208982094161507, 'ndcg@50': 0.5277202114549242}


,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.06050,0.09788,0.13197,0.18067,0.27392
baseline,0.24466,0.36097,0.42827,0.48209,0.52772


In [36]:
test_df['user_id'].n_unique()

1785

### III.III. Реализация обучения градиентного бустинга (2 из 4 баллов)

Обучите ранкер на train части датасета. В качестве модели можете использовать любую из [Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/). Обучать можно как Ranker, так и Classifier, так и Regressor. Поэкспериментируйте. Посчитайте метрики на val части и подберите гиперпараметры (можете взять разные временные срезы, чтобы не заоверфиттиться под один).

Обучите итоговую модель на train + val, замерьте качество на test и сравните с бейзлайном.

Sanity check. Обратите внимание, что если вы считаете бейзлайн по фиче из датасета, то фича, по которой вы считаете бейзлайн, обязательно должна присутствовать как фича для ранкера. Если ранкер при использовании этой фичи показывает результаты хуже, чем бейзлайн, то что-то с вашим ранкером не так.

In [ ]:
# на всякий случай добавляю скор популярности
item_pop = (
    train_df
    .group_by("item_id")
    .agg(pl.col("label").sum().alias("f_num__global_pop_score"))
)

train_df     = train_df.join(item_pop, on="item_id", how="left").with_columns(pl.col("f_num__global_pop_score").fill_null(0))
val_df       = val_df.join(item_pop, on="item_id", how="left").with_columns(pl.col("f_num__global_pop_score").fill_null(0))
train_val_df = train_val_df.join(item_pop, on="item_id", how="left").with_columns(pl.col("f_num__global_pop_score").fill_null(0))
test_df      = test_df.join(item_pop, on="item_id", how="left").with_columns(pl.col("f_num__global_pop_score").fill_null(0))

features = [col for col in train_df.columns if col.startswith("f_")]
categorical_features = [col for col in features if col.startswith("f_cat__")]
print(f"Total features: {len(features)}, categorical: {len(categorical_features)}")


Total features: 303, categorical: 0


In [38]:
def train_evaluate_model(params, train_df, val_df, features, categorical_features, use_early_stopping=True):
    """
    Train LGBMRanker on train_df, evaluate on val_df.

    Args:
        use_early_stopping: If True, use val_df for early stopping during fit.
            Set to False for the final model trained on train+val to avoid
            leaking test data into the stopping criterion.
    Returns (model, metrics_dict).
    """
    import lightgbm as lgbm

    # Sort by session_id (required for LGBMRanker group sizes)
    train_sorted = train_df.sort("session_id")
    val_sorted   = val_df.sort("session_id")

    X_train = train_sorted.select(features).to_pandas()
    y_train = train_sorted["label"].to_numpy()
    group_train = (
        train_sorted
        .group_by("session_id")
        .agg(pl.len().alias("cnt"))
        .sort("session_id")["cnt"]
        .to_numpy()
    )

    X_val = val_sorted.select(features).to_pandas()
    y_val = val_sorted["label"].to_numpy()
    group_val = (
        val_sorted
        .group_by("session_id")
        .agg(pl.len().alias("cnt"))
        .sort("session_id")["cnt"]
        .to_numpy()
    )

    cat_indices = [features.index(c) for c in categorical_features if c in features]

    model = lgbm.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=params.get("n_estimators", 300),
        learning_rate=params.get("learning_rate", 0.05),
        num_leaves=params.get("num_leaves", 63),
        max_depth=params.get("max_depth", -1),
        min_child_samples=params.get("min_child_samples", 20),
        subsample=params.get("subsample", 0.8),
        colsample_bytree=params.get("colsample_bytree", 0.8),
        reg_alpha=params.get("reg_alpha", 0.0),
        reg_lambda=params.get("reg_lambda", 0.0),
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    fit_kwargs = {
        "categorical_feature": cat_indices if cat_indices else "auto",
    }
    if use_early_stopping:
        fit_kwargs["eval_set"] = [(X_val, y_val)]
        fit_kwargs["eval_group"] = [group_val]
        fit_kwargs["eval_at"] = [5, 10]
        fit_kwargs["callbacks"] = [lgbm.early_stopping(50, verbose=False), lgbm.log_evaluation(period=-1)]

    model.fit(X_train, y_train, group=group_train, **fit_kwargs)

    # Predict on val and compute metrics
    val_scores = model.predict(X_val)
    val_with_score = val_sorted.with_columns(pl.Series("score", val_scores))
    val_grouped = (
        val_with_score
        .with_columns(pl.lit(np.random.default_rng(42).random(val_with_score.height)).alias("random_noise"))
        .group_by("session_id")
        .agg([
            pl.struct("item_id").sort_by(["score", "random_noise"], descending=[True, True]).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ])
    )
    metrics = evaluate_ranker(val_grouped)
    return model, metrics


In [39]:
def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    _, metrics = train_evaluate_model(params, train_df, val_df, features, categorical_features)
    return metrics.get("ndcg@5", 0.0)

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nBest ndcg@5: {study.best_value:.5f}")
print(f"Best params: {study.best_params}")

# Train final model on train+val with use_early_stopping=False to avoid
# leaking test data into the stopping criterion (test used only for final predict).
best_params = study.best_params.copy()
best_model, _ = train_evaluate_model(
    best_params,
    train_val_df,
    test_df,  # passed for val metrics only; early stopping is disabled
    features,
    categorical_features,
    use_early_stopping=False,
)

# Evaluate on test
test_sorted = test_df.sort("session_id")
X_test = test_sorted.select(features).to_pandas()
test_scores = best_model.predict(X_test)

test_with_score = test_sorted.with_columns(pl.Series("score", test_scores))
test_grouped = (
    test_with_score
    .with_columns(pl.lit(np.random.default_rng(42).random(test_with_score.height)).alias("random_noise"))
    .group_by("session_id")
    .agg([
        pl.struct("item_id").sort_by(["score", "random_noise"], descending=[True, True]).alias("preds"),
        pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
    ])
)
ranker = evaluate_ranker(test_grouped)
print("\nRanker on test:", ranker)

RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(ranker, index=["ranker"]),
])

RESULTS.style.format(precision=5).background_gradient(cmap="Blues")


  0%|          | 0/100 [00:00<?, ?it/s]


Best ndcg@5: 0.37969
Best params: {'n_estimators': 287, 'learning_rate': 0.08633635023135917, 'num_leaves': 53, 'min_child_samples': 20, 'subsample': 0.727430401598145, 'colsample_bytree': 0.9273664711709627, 'reg_alpha': 2.226560018976442, 'reg_lambda': 0.006937736203225271}

Ranker on test: {'ndcg@1': 0.2678404695211418, 'ndcg@5': 0.3737080222571756, 'ndcg@10': 0.43876781470706544, 'ndcg@20': 0.4936489120422654, 'ndcg@50': 0.5372229669782029}


,ndcg@1,ndcg@5,ndcg@10,ndcg@20,ndcg@50
best,1.00000,1.00000,1.00000,1.00000,1.00000
worst,0.06050,0.09788,0.13197,0.18067,0.27392
baseline,0.24466,0.36097,0.42827,0.48209,0.52772
ranker,0.26784,0.37371,0.43877,0.49365,0.53722


Опишите полученный результат - получилось ли обогнать бустингом baseline? Почему?

> **Вывод по результатам ранжирования**

> **Превосходство персонализации над популярностью:** Бустинг показал ndcg@5 = 0.37371 против 0.36097 у бейзлайна.  Это доказывает, что признаки из Datamart (конверсии, история конкретного пользователя, ценовые предпочтения) несут  персонализированный сигнал, который глобальная популярность не способна уловить.

> **Сила бейзлайна:** Несмотря на победу ML-модели, TopPopular выдаёт очень достойные цифры (что типично для e-commerce: популярные товары кликают и покупают чаще всего). Это подчёркивает важность добавления фичи `f_num__global_pop_score` в сам бустинг — модель опирается на тренд популярности, корректируя его под конкретного пользователя (например, через конверсию пользователя по категории или историю кликов).
